In [34]:
import pandas as pd
import numpy as np
import ast
import pickle
import nltk

from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/khushi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [36]:
# Load datasets
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

In [37]:
print(movies.shape)
print(credits.shape)

(4803, 20)
(4803, 4)


In [38]:
movies = movies.merge(credits, on='title')

In [39]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

In [40]:
movies.dropna(inplace=True)

In [41]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [67]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [42]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [43]:
movies['genres'] = movies['genres'].apply(convert)

In [44]:
movies['keywords'] = movies['keywords'].apply(convert)

In [45]:
def convert_cast(text):
    L = []

    counter = 0

    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [46]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [47]:
def fetch_director(text):
    L = []

    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [48]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [49]:
print(movies['genres'].iloc[0])
print(movies['keywords'].iloc[0])
print(movies['cast'].iloc[0])
print(movies['crew'].iloc[0])

['Action', 'Adventure', 'Fantasy', 'Science Fiction']
['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']
['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
['James Cameron']


In [50]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [51]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])

In [52]:
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])

In [53]:
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])

In [54]:
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [55]:
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [56]:
new_df = movies[['movie_id', 'title', 'tags']].copy()

In [57]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [58]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [59]:
ps = PorterStemmer()

In [60]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [62]:
new_df['tags'] = new_df['tags'].apply(stem)

In [63]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


In [64]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [65]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [66]:
vectors.shape

(4806, 5000)

In [68]:
similarity = cosine_similarity(vectors)

In [69]:
similarity.shape

(4806, 4806)

In [70]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ])

In [71]:
def recommend(movie):
    # Find the index of the movie
    movie_index = new_df[new_df['title'] == movie].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort by similarity (highest first), excluding the movie itself
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    # Print top 5 recommendations
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [72]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [73]:
recommend("Batman Begins")

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [74]:
recommend("The Dark Knight")

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman


In [75]:
recommend("Iron Man")

Iron Man 3
Iron Man 2
Avengers: Age of Ultron
The Avengers
Captain America: Civil War


In [76]:
recommend("The Avengers")

Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
Captain America: The First Avenger
Iron Man


In [77]:
import os

os.makedirs("../models", exist_ok=True)

pickle.dump(new_df, open("../models/movie_list.pkl", "wb"))

print("Movie list saved successfully!")

Movie list saved successfully!


In [78]:
pickle.dump(similarity, open("../models/similarity.pkl", "wb"))

print("Similarity matrix saved successfully!")

Similarity matrix saved successfully!


In [79]:
import os

print(os.listdir("../models"))

['similarity.pkl', 'movie_list.pkl']
